<a href="https://colab.research.google.com/github/dimamagdenko07-spec/python-data-analysis-olist/blob/main/notebooks/python_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
time = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
to_read = time + ['order_id', 'customer_id', 'order_status']


Задание 1

In [ ]:
#Считываем все, так как 3 нужные мы выбрали, а все остальные содержат время
orders = pd.read_csv('olist_orders_dataset.csv', usecols=to_read)
print(orders.columns)
orders[time] = orders[time].apply(pd.to_datetime, errors = 'coerce')
#deep = True значит, что при получении указателя:
#Мы возвращаем не память, выделенную под него, а то, что внутри
memory_status_before = orders['order_status'].memory_usage(deep=True)
orders['order_status'] = orders['order_status'].astype('category')
memory_status_after = orders['order_status'].memory_usage(deep=True)
print(f"Раньше: {memory_status_before}, теперь: {memory_status_after}")

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Раньше: 5766064, теперь: 100333


Изначально любые данные в таблице хранятся в формате Python Object, поэтому строчные значения занимают очень много места в памяти. Так как статусы у нас могут быть не большого количества типов, то мы можем сделать тип "category", который позволяет хранить все названия, которые есть (3 штуки), а все данные типа category ссылаются на эти данные => представляют из себя число => занимают гораздо меньше памяти

Задание 2

In [ ]:
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
#Заполняет все значения NaN на No title/No message
reviews['review_comment_title'] = reviews['review_comment_title'].fillna("No title")
reviews['review_comment_message'] = reviews['review_comment_message'].fillna("No message")
#Приводит все буквы в комментах к нижнему регистру
reviews['review_comment_message'] = reviews['review_comment_message'].str.lower()
reviews['review_comment_title'] = reviews['review_comment_title'].str.lower()
#Заменяет все спец символы на пустоту. regex = True => позвоняет заменять несколько символов сразу
reviews['review_comment_message'] = reviews['review_comment_message'].str.replace(r'\n|\r', '', regex=True)
reviews['review_comment_title'] = reviews['review_comment_title'].str.replace(r'\n|\r', '', regex=True)
#Сортирует данные по времени первого коммента у человека. Удаляет все кроме первого(самого нового)
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews = reviews.sort_values(by=['order_id', 'review_creation_date'], ascending=[True, False])
reviews = reviews.drop_duplicates(subset=['order_id'], keep='first')
print(reviews)

                              review_id                          order_id  \
51963  97ca439bc427b48bc1cd7177abe71365  00010242fe8c5a6d1ba2dd792cb16214   
27823  7b07bacd811c4117b742569b04ce3580  00018f77f2f0320c557190d7a144bdd3   
4218   0c5b33dea94867d1ac402749e5438e8b  000229ec398224ef6ca0657da4fc703e   
38844  f4028d019cb58564807486a6aaf33817  00024acbcdf0a6daa1e931b038114c75   
55676  940144190dcba6351888cafa43f3a3a5  00042b26cf59d7ce69dfabb4e55b4fd9   
...                                 ...                               ...   
48178  9185f849f32d82e216a4e025e0c50f5c  fffc94f6ce00a00581880bf54a75a037   
21684  be803f6a93d64719fd685c1cc610918a  fffcd46ef2263f404302a634eb57f7eb   
37509  dbdd81cd59a1a9f94a10a990b4d48dce  fffce4705a9662cd70adb13d4a31832d   
50462  fba117c9ac40d41ca7be54741f471303  fffe18544ffabc95dfada21779c9644f   
92457  b2700869a37f1aafc9dda829dc2f9027  fffe41c64501cc87c801fd61db3f6244   

       review_score review_comment_title  \
51963             5            

Задание 3

In [ ]:
customers = pd.read_csv("olist_customers_dataset.csv", dtype = {"customer_zip_code_prefix" : str})
#Берём первую букву города
customers['city_initial'] = customers['customer_city'].str[0]
#Делаю выборку по условиям для таблицы
filter = customers.loc[
    (customers['customer_state'] == 'SP') &
    (customers['customer_zip_code_prefix'].str.startswith('01'))
]
#Заменяю индексы с 0,1..n на customer_id
customers_id = customers.set_index(customers['customer_id'])
id = '06b8999e2fba1a1fbc88172c00ba8bc7'
#Выдаю инфу по посетителю по id
customer = customers_id.loc[id]
print(filter)

                            customer_id                customer_unique_id  \
2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
292    d703cdda08125e2b263b92746af0f23c  4669d94e1e35f2493af76862ff85beba   
300    dcb6e7bc4e65d815025d0aa14349a4ca  a1864cdd58debf5031958069ee937742   
333    ee23d61fedfc4227bb0e188654c28d08  494c6658198b47787205a06421bb9403   
364    3f5939e06e89efd9e9ca8ef3ab9d68c7  6b7e1090d106aaee31d5ed12d2bbc2af   
...                                 ...                               ...   
98967  233f73a4bc9db336b82cb23cf5ff2701  03d32fb1ecfa4d9f7bbb6918614662ae   
99184  8300b253882166d54df794107d32f0bc  a09a3b1eec6b18cc3e3235d93ceeed29   
99214  a717d4cccf6200ea92a5047cb00364db  9824bf31431205c6c2e87f32c0fd0258   
99408  f6c6d3e1e20969a5eed982163f959719  fb354969e06f2093c0083cbfbb91864e   
99423  c6ece8a5137f3c9c3a3a12302a19a2ac  aaf22868003377e859049dcf5f0b3fdf   

      customer_zip_code_prefix customer_city customer_state city_initial  
